# Sequence classification with Conv1D

Classify synthetic waveforms (sine, square, triangle) using 1D convolutions.
This example demonstrates Conv1D, MaxPool1D, and dropout.

```
Conv1D(1->4, k=3) -> ReLU -> MaxPool1D(2) ->
Conv1D(4->8, k=3) -> ReLU -> MaxPool1D(2) ->
Dropout(0.5) -> Linear(48->3) (raw logits; loss applies log_softmax)
```

**CLI equivalent:** `make example-seq-classify` (1000 epochs)


## Architecture

The same type-safe dimension chain as the CNN notebook, but in 1D.
`ConvOutDim` and `PoolOutDim` work identically for 1D and 2D.


In [ ]:
:t conv1d

In [ ]:
:t maxPool1d

In [ ]:
:t dropout

## Dimension chain

Input: 32 timesteps, 1 channel (flat dim = 32).

| Layer | Length | Channels | Flat dim |
|-------|--------|----------|----------|
| Input | 32 | 1 | 32 |
| Conv1D(k=3) | 30 | 4 | 120 |
| MaxPool(2) | 15 | 4 | 60 |
| Conv1D(k=3) | 13 | 8 | 104 |
| MaxPool(2) | 6 | 8 | 48 |
| Dropout(0.5) | - | - | 48 |
| Linear | - | - | 3 |

All flat dims stay under 120 to avoid Idris 2's Peano Nat
type-checking ceiling (~1000).


## Data: synthetic waveforms

Three classes with random frequency and phase:
- **Sine**: smooth oscillation
- **Square**: binary high/low
- **Triangle**: linear ramps

Fresh data is generated each epoch, so the model must learn general
waveform features rather than memorizing specific samples.


## Model construction

The compiled example chains the pipeline into a `Seq` (`Example/SeqClassify.idr`):

```idris
mkModel : Init Model
mkModel = do
  c1 <- conv1d {inC = InC} {outC = C1} {len = SeqLen}   {kL = K} {pad = 0}
  c2 <- conv1d {inC = C1}  {outC = C2} {len = Pool1Out} {kL = K} {pad = 0}
  l  <- linear {i = AfterPool2} {o = NumClasses}
  pure (c1 ~~> reluA
           ~~> maxPool1d {c = C1} {len = Conv1Out} {poolK = 2} {str = 2}
           ~~> c2 ~~> reluA
           ~~> maxPool1d {c = C2} {len = Conv2Out} {poolK = 2} {str = 2}
           ~~> dropout 0.5
           ~~> l ~~> Nil)
```

The next cell builds the first conv layer live; the waveform data generator
is in the compiled example.


In [ ]:
:exec run (do {
  m <- runInitL (conv1d {inC=1} {outC=4} {len=32} {kL=3} {pad=0}
                  {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  discard m; liftIO1 (putStrLn "conv1d built (1x32 -> 4x30).") })

## Training

Generated waveforms are collated into batches C-side and handed to
`fitSupervised` with Adam under `NormClip 1.0` and early stopping:

```idris
opt <- adam cfg.lr ({ clip := NormClip 1.0 } defaultOpts)
let bs = batched {b = BatchSize} {i = InputDim} {o = NumClasses} (generate mkSample)

(MkBang (epochsDone, finalLoss) # trained) <-
  fitSupervised opt nllLossL bs (patienceConfig cfg.epochs cfg.patience) model
```

Run via CLI: `make example-seq-classify SEQ_CLASSIFY_ARGS="--epochs 1000"`


## PyTorch comparison

```python
model = nn.Sequential(
    nn.Conv1d(1, 4, 3), nn.ReLU(), nn.MaxPool1d(2),
    nn.Conv1d(4, 8, 3), nn.ReLU(), nn.MaxPool1d(2),
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(48, 3)
)
```

Note that PyTorch needs `nn.Flatten()` between the conv and linear layers.
In idris-ml, the flat dimension `48 = 8 * 6` is already a type-level Nat,
so no flatten is needed (or possible to get wrong).

See `pytorch/torch_ref/scripts/seq_classify.py` for the full reference.


Next: [BERT](bert.ipynb) — loading a real HuggingFace checkpoint and running
a typed forward pass on it.
